<img src="http://imgur.com/1ZcRyrc.png" style="float: left; margin: 20px; height: 55px">

# Introduction to Gridsearching Hyperparameters

_Authors: Kiefer Katovich (SF), David Yerrington (SF)_

---

![](https://snag.gy/aYcCt2.jpg)

### Learning Objective
- Describe what the terms gridsearch and hyperparameter mean.
- Build a gridsearching procedure from scratch.
- Apply sklearn's `GridSearchCV` object with basketball data to optimize a KNN model.
- Use and evaluate attributes of the gridsearch object.
- Describe the pitfalls of searching large hyperparameter spaces.

### Lesson Guide
- [What is "grid searching"? What are "hyperparameters"?](#intro)
- [Basketball Data](#basketball-data)
- [Fitting a Default KNN](#fit-knn)
- [Searching for the Best Hyperparameters](#searching)
    - [Grid Search Pseudocode](#pseudocode)
    - [Using `GridSearchCV`](#gscv)
- [A Caution on Grid Searching](#caution)
- [Independent Practice: Grid Searching Regularization Penalties with Logistic Regression](#practice)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('fivethirtyeight')

%matplotlib inline

<a id='intro'></a>

## What is "grid searching"? What are "hyperparameters"?

---

Models often have built-in quantities that we can use to fine-tune our results. For example, when we choose a linear regression, we may decide to add a penalty to the loss function such as the Ridge or the Lasso. The Ridge and Lasso have a hyperparameter $\alpha$, which we specify ourselves.

> **Parameters** are quantities that a model can learn (for example, coefficients of linear regression).

> **Hyperparameters** are quantities that a model cannot learn, that we must decide on ourselves (for example, the penalty term $\alpha$).

Models often have **multiple hyperparameters**. For example, in a kNN model, we have $k$, the number of neighbors to use, but we also have other chocies such as _weights_ or _distance metric_ (usually these are set to "uniform" and "Euclidean", but they can be changed!).

However, different values for hyperparameters result in different quality fits. **We want to find the optimal values for our hyper parameters.**

![](imgs/grid.jpg)

**One method of searching for the optimal set of hyperparameters is called gridsearching.**

Gridsearching gets its name from the fact that we are searching over a "grid" of parameters. For example, imagine the `n_neighbors` hyperparameters on the x-axis and `weights` on the y-axis, and we need to test all points on the grid.

**Gridsearching uses cross-validation internally to evaluate the performance of each set of hyperparameters.** More on this later.

<a id='basketball-data'></a>

<a id='basketball-data'></a>

## Basketball Data

---

To explore the process of gridsearching over sets of hyperparameters, we will use some basketball data. The data below has statistics for 4 different seasons of NBA basketball: 2013-2016.
- This data includes aggregate statistical data for each game. 
- The data of each game is aggregated by match for all players.
- Scraped from http://www.basketball-reference.com

Many of the columns in the dataset represent the mean of a statistic across the last 10 games, for example. Non-target statistics are for *prior* games, they do not include information about player performance in the current game.

**We are interested in predicting whether the home team will win the game or not.** This is a classification problem.


### Load the data and create the target and predictor matrix
- The target will be a binary column of whether the home team wins.
- The predictors should be numeric statistics columns.

Exclude these columns from the predictor matrix:

    ['GameId','GameDate','GameTime','HostName',
     'GuestName','total_score','total_line','game_line',
     'winner','loser','host_wins','Season']


In [ ]:
data = pd.read_csv('../datasets/basketball_data.csv')

In [ ]:
# Create a binary int column to represent host's win/loss

### Create the training and testing data
- Test data should be the 2016 season data, train data will be the previous seasons.
- Make sure to standardize your predictor matrix (easiest to do prior to splitting the data into training and testing)!

<a id='fit-knn'></a>

## Fitting the Default KNN

---

Below we fit a default `KNeighborsClassifier` to predict win vs. not on the training data, then score it on the testing data. 

Make sure to compare your score to the baseline accuracy.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

<a id='searching'></a>

## Searching for the Best Hyperparameters

--

Our default KNN performs quite poorly on the test data. But what if we changed the number of neighbors? The weighting? The distance metric?

These are all hyperparameters of the KNN. How would we do this manually? We would need to evaluate on the training data the set of hyperparameters that perform best, and then use this set of hyperparameters to fit the final model and score on the testing set.

<a id='pseudocode'></a>
### Gridsearch pseudocode for our KNN

```python
accuracies = {}
for k in neighbors_to_test:
    for w in weightings_to_test:
        for d in distance_metrics_to_test:
            hyperparam_set = (k, w, d)
            knn = KNeighborsClassifier(n_neighbors=k, weights=w, metric=d)
            cv_accuracies = cross_val_score(knn, X_train, y_train, cv=5)
            accuracies[hyperparam_set] = np.mean(cv_accuracies)
```

In the pseudocode above, we would find the key in the dictionary (a hyperparameter set) that has the largest value (mean cross-validated accuracy).



<a id='gscv'></a>
### Using `GridSearchCV`

This would be an annoying process to have to do manually. Luckily sklearn comes with a convenience class for performing gridsearch:

```python
from sklearn.model_selection import GridSearchCV
```

The `GridSearchCV` has a handful of important arguments:

| Argument | Description |
| --- | ---|
| **`estimator`** | The sklearn instance of the model to fit on |
| **`param_grid`** | A dictionary where keys are hyperparameters for the model and values are lists of values to test |
| **`cv`** | The number of internal cross-validation folds to run for each set of hyperparameters |
| **`n_jobs`** | How many cores to use on your computer to run the folds (-1 means use all cores) |
| **`verbose`** | How much output to display (0 is none, 1 is limited, 2 is printouts for every internal fit) |


Below is an example for how one might set up the gridsearch for our KNN:

```python
knn_parameters = {
    'n_neighbors':[1,3,5,7,9],
    'weights':['uniform','distance']
}

knn_gridsearcher = GridSearchCV(KNeighborsClassifier(), knn_parameters, verbose=1)
knn_gridsearcher.fit(X_train, y_train)
```

**Try out the sklearn gridsearch below on the training data.**

In [ ]:
from sklearn.model_selection import GridSearchCV

### Read the docs!
[GridSearchCV documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)

<a id='gs-results'></a>
### Examining the Results of the Grid Search

Once the grid search has fit (this can take awhile!) we can pull out a variety of information and useful objects from the gridsearch object, stored as attributes:

| Property | Use |
| --- | ---|
| **`results.param_grid`** | Displays parameters searched over. |
| **`results.best_score_`** | Best mean cross-validated score achieved. |
| **`results.best_estimator_`** | Reference to model with best score.  Is usable / callable. |
| **`results.best_params_`** | The parameters that have been found to perform with the best score. |
| **`results.grid_scores_`** | Display score attributes with corresponding parameters. | 

**Print out the best score found in the search.**

**Print out the set of hyperparameters that achieved the best score.**

**Assign the best fit model (`best_estimator_`) to a variable and score it on the test data.**

Compare this model to the baseline accuracy and your default KNN.

In [ ]:
print('baseline:', np.mean(y_test))
print('default KNN:', knn.score(X_test, y_test))

**Let's see.... _EVERYTHING!_**

In [ ]:
gs_df = pd.DataFrame(knn_gridsearch.cv_results_)
gs_df = gs_df[gs_df['param_metric'] == 'euclidean']
gs_df.plot(x='param_n_neighbors', y='mean_test_score')

<a id='caution'></a>

## A Word of Caution on Grid Searching

---

Sklearn models often have many options/hyperparameters with many different possible values. It may be tempting to search over a wide variety of them. In general, this is not wise.

Remember that **gridsearch searches over all possible combinations of hyperparamters in the paramter dictionary!**

The KNN model class takes a wider range of options during instantiation than we have explored. Imagine that we had this as our parameter dictionary:

```python
parameter_grid = {
    'n_neighbors': range(1, 151),
    'weights': ['uniform', 'distance', custom_function],
    'algorithm': ['ball_tree', 'kd_tree', 'brute', 'auto'],
    'leaf_size': range(1, 152),
    'metric': ['minkowski', 'euclidean'],
    'p': [1, 2]
}
```

**How many different combinations will need to be tested?

| Parameter | Number of Chosen Values |
| --- | --- |
| **n_neighbors** | 150 |
| **weights** | 3 |
| **algorithm** | 4 |
| **leaf_size** | 151 |
| **metric** | 2 |
| **p** | 2 |
| <br>_150 \* 3 \* 4 \* 151 \* 2 \* 2 = n combinations_ <br><br>| _1,087,200_ |

And that is over a million tests *before we even consider the number of cross-validation folds!*

If you're not careful, gridsearching can quickly blow up. A lot of the hyperparameters we put in the dumb example above are either redundant or not useful.

> **It is extremely important to understand what the hyperparameters do and think critically about what ranges are useful and relevant to your model!**


## Another Option:
`from sklearn.model_selection import RandomizedSearchCV`

# Wrapping it all up in a Pipeline
![](imgs/pipe.png)

## But first: A Scikit-Learn Aside
**DID YOU KNOW?** Scikit-Learn has two types of classes inside? **Estimators** and **transformers**. We've seen several examples of each so far. Here's the rundown of their differences:

### Scikit-Learn Estimators
Estimators are essentially _models_. They fit this format:

```python
model = MyCoolModel(params)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
```

Estimators have a **fit** and **predict** method.

### Scikit-Learn Transformers
Transformers are not models. They transform your data using similar syntax to estimators. They work like this:

```python
trf = MyCoolTransformer(params)
trf.fit(X_train)
X_transformed = trf.transform(X_train)
```

Instead of predict, they have **fit** and **transform** methods.  In fact, since you fit and transform together so often, they have a shortcut:

```python
trf = MyCoolTransformer(params)
X_transformed = trf.fit_transform(X_train)
```

We've seen a few transformers `StandardScaler()` and `PolynomialFeatures()`. There's also `OneHotEncoder()` for dummy encoding and `LabelEncoder()` for factorizing variables. Later we'll see `PCA()`, which is also a transformer.

### Pipelines
Often, we'll want to apply several transformers to a dataset, and then finally build a model. If you do all of these steps independently, your code when predicting on test data can be messy. It'll also be prone to errors. Luckily, we'll have **pipelines**! Let's code one up.

In [ ]:
# Import Pipeline
from sklearn.pipeline import Pipeline

In [ ]:
# Build a StandardScaler + kNN pipeline

# Fit

In [ ]:
# Score, predict work!

In [ ]:
# Get params - yes, you can GridSearchCV over these!
# Notice the naming convention of pipe arguments.

<a id='practice'></a>

## Practice: Grid Search Regularization Penalties with Logistic Regression

---

Logistic regression models can also apply the Lasso and Ridge penalties. The `LogisticRegression` class takes these regularization-relevant hyperparameters:

| Argument | Description |
| --- | ---|
| **`penalty`** | `'l1'` for Lasso, `'l2'` for Ridge |
| **`solver`** | Must be set to `'liblinear'` for the Lasso penalty to work. |
| **`C`** | The regularization strength. Equivalent to `1./alpha` |

**You should:**
1. Fit and validate the accuracy of a default logistic regression on the basketball data.
- Perform a gridsearch over different regularization strengths and Lasso and Ridge penalties.
- Compare the accuracy on the test set of your optimized logistic regression to the baseline accuracy and the default model.
- Look at the best parameters found. What was chosen? What does this suggest about our data?
- Look at the (non-zero, if Lasso was selected as best) coefficients and associated predictors for your optimized model. What appears to be the most important predictors of winning the game?


In [ ]:
from sklearn.linear_model import LogisticRegression

# REMEMBER:
# We already have X_train, X_test unscaled! No need to redefine.

lr = LogisticRegression()
lr.fit(X_train, y_train)

print(lr.score(X_test, y_test))

In [ ]:
print(np.mean(y_test))

In [ ]:
# Set up the parameters. Looking at C regularization strengths on a log scale.
gs_params = {
    'penalty': ['l1', 'l2'],
    'C': np.logspace(-4, 0, 40)
}

lr_gridsearch = GridSearchCV(
    LogisticRegression(solver='liblinear'),
    gs_params,
    cv=5,
    n_jobs=4,
    verbose=1,
    return_train_score=False
)

In [ ]:
lr_gridsearch = lr_gridsearch.fit(X_train, y_train)

In [ ]:
# Best score on the training data:
lr_gridsearch.best_score_

In [ ]:
# Best parameters on the training data:
lr_gridsearch.best_params_

In [ ]:
# Lasso was chosen: this indicates that maybe unimportant (noise) variables
# is more of an issue in our data than multicollinearity.

In [ ]:
# assign the best estimator to a variable:
best_lr = lr_gridsearch.best_estimator_

In [ ]:
# Score it on the testing data:
best_lr.score(X_test, y_test)

In [ ]:
gs_df = pd.DataFrame(lr_gridsearch.cv_results_)
gs_df = gs_df[gs_df['param_penalty'] == 'l2']

In [ ]:
gs_df.plot(x='param_C', y='mean_test_score')